# 🎓 Machine Learning Project: Personalized AI Tutor via LLM & Reinforcement Learning
## 📌 Step 03: Data Cleaning & Preprocessing

---

### 📖 Objective:
Clean missing values, filter sparse skills ($< 20$ records), standardize binary target variables, clip latency outliers, and sort interactions chronologically per student.


### Cell 1: Import Libraries


In [ ]:
# Cell 1: Import Libraries
import os
import pandas as pd
import numpy as np

print("✅ Libraries Imported Successfully")


### Cell 2: Load Raw Dataset


In [ ]:
# Cell 2: Load Raw Dataset
cur_dir = os.path.abspath(".")
DATA_DIR = os.path.join(cur_dir, "data") if os.path.exists("data") else cur_dir
RAW_DATA_PATH = os.path.join(DATA_DIR, "skill_builder_data_corrected.csv")
PROCESSED_PATH = os.path.join(DATA_DIR, "assistments_processed.csv")

df_raw = pd.read_csv(RAW_DATA_PATH, low_memory=False, encoding='latin1')
print(f"Loaded Raw Records: {len(df_raw):,}")
print("✅ Dataset Loaded Successfully")


### Cell 3: Data Cleaning & Standardization Pipeline


In [ ]:
# Cell 3: Execute Cleaning Pipeline
user_col, skill_col, correct_col = "user_id", "skill_name", "correct"

# 1. Filter missing skill and target labels
df_clean = df_raw.dropna(subset=[skill_col, correct_col]).copy()

# 2. Standardize target to binary {0, 1}
df_clean[correct_col] = df_clean[correct_col].astype(float).round().astype(int)
df_clean = df_clean[df_clean[correct_col].isin([0, 1])]

# 3. Clean response latencies (clip 2s to 300s)
if "ms_first_response" in df_clean.columns:
    df_clean["response_time_sec"] = (df_clean["ms_first_response"] / 1000.0).clip(2.0, 300.0)
else:
    df_clean["response_time_sec"] = 20.0

# 4. Clean hints
if "hint_count" in df_clean.columns:
    df_clean["hint_count"] = df_clean["hint_count"].fillna(0).astype(int).clip(0, 10)
else:
    df_clean["hint_count"] = 0

# 5. Filter sparse skills with under 20 interactions
freq_skills = df_clean[skill_col].value_counts()
df_clean = df_clean[df_clean[skill_col].isin(freq_skills[freq_skills >= 20].index)]

# 6. Sort chronologically per student
sort_keys = [user_col]
if "order_id" in df_clean.columns:
    sort_keys.append("order_id")
df_clean = df_clean.sort_values(by=sort_keys).reset_index(drop=True)

# Save clean dataset
df_clean.to_csv(PROCESSED_PATH, index=False)
print(f"Preprocessed dataset saved to: {PROCESSED_PATH}")
print(f"Final Clean Shape: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")


### Cell 4: Clean Data Inspection (`df_clean.head()` & `df_clean.info()`)


In [ ]:
# Cell 4: Inspection
print("--- Clean DataFrame Information: ---")
df_clean.info()
print("\n--- First 5 Clean Records: ---")
display(df_clean.head())
print("✅ Step 03 Completed Successfully!")
